In [20]:
import pandas as pd 
import numpy as np 
import sklearn as sk
import seaborn as sns
import os 
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.preprocessing import MaxAbsScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
import os
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"

In [21]:
data_path='/mnt/cold2/snaketree/prj/PPH/local/share/data/saver_mat'
train_id=['CRC1502','CRC1620','CRC1139']
tratt_cercato=['cetux','CTX72h']
train=pd.DataFrame()
for file in os.listdir(data_path):
    sample_name=str.split(file,sep='_')[4]
    trattamento=str.split(file,sep='_')[5]
    if sample_name in train_id and trattamento in tratt_cercato:
        print(file)
        data=pd.read_csv(os.path.join(data_path,file),header=0,index_col=0)
        data=data.T
        data['sample']=sample_name
        data['cell_id']=data.index
        data['trattamento']=trattamento
        data.reset_index(drop=True,inplace=True)
        train=pd.concat([train,data])

filtered_annotated_saver_ribomito_CRC1502_cetux_1_log2_pc1_cpm.csv
filtered_annotated_saver_ribomito_CRC1139_cetux_1_log2_pc1_cpm.csv
filtered_annotated_saver_ribomito_CRC1620_cetux_1_log2_pc1_cpm.csv


In [22]:
from func_NT_cetux import *
df=train
df_clean, dup = strip_prefix_from_genes(df, meta_cols=("cell_id","sample","trattamento"), sep=":", on_duplicate="first")

In [23]:
hvg=pd.read_csv('bbknn_cetux_KRAS/npcs=20__nwb=3__trim=None__res=0.3/highly_variable_genes.csv',header=0,index_col=False)
to_keep=list(hvg['gene'])
to_keep = to_keep + ['sample', 'cell_id', 'trattamento']

In [24]:
df_clean = df_clean.loc[:,to_keep]

In [25]:
clu_path='bbknn_cetux_KRAS/npcs=20__nwb=3__trim=None__res=0.3/obs_data_with_umap.csv'
meta_data=pd.read_csv(clu_path,header=0)

In [26]:
meta_data

,cell_id,sample,trattamento,uid,S_score,G2M_score,phase,leiden_r0.3,umap_1,umap_2
0,AAACCCAAGGTGCGAT.1,CRC1502,cetux,CRC1502cetux|AAACCCAAGGTGCGAT.1|cetux,-0.064950,2.195363,G2M,5,0.397067,5.521682
1,AAACCCACAACGGTAG.1,CRC1502,cetux,CRC1502cetux|AAACCCACAACGGTAG.1|cetux,-0.912317,-1.891038,G1,1,10.038173,5.172250
2,AAACCCACAAGGTCAG.1,CRC1502,cetux,CRC1502cetux|AAACCCACAAGGTCAG.1|cetux,1.049680,-0.334879,S,3,3.651914,4.905559
3,AAACCCACATTCAGGT.1,CRC1502,cetux,CRC1502cetux|AAACCCACATTCAGGT.1|cetux,0.363655,3.044909,G2M,5,-0.935656,5.826066
4,AAACCCAGTATTGGCT.1,CRC1502,cetux,CRC1502cetux|AAACCCAGTATTGGCT.1|cetux,-0.835467,-1.460480,G1,0,7.646122,2.865105
...,...,...,...,...,...,...,...,...,...,...
12942,TTTGGTTTCGTGGGTC.1,CRC1620,cetux,CRC1620cetux|TTTGGTTTCGTGGGTC.1|cetux,-1.530862,-1.665001,G1,0,8.099384,3.335701
12943,TTTGTTGAGACAACAT.1,CRC1620,cetux,CRC1620cetux|TTTGTTGAGACAACAT.1|cetux,0.657491,1.482736,G2M,2,-2.352744,5.552454
12944,TTTGTTGAGTAGGATT.1,CRC1620,cetux,CRC1620cetux|TTTGTTGAGTAGGATT.1|cetux,-1.637527,-1.463938,G1,4,11.430054,2.986914
12945,TTTGTTGCACCATTCC.1,CRC1620,cetux,CRC1620cetux|TTTGTTGCACCATTCC.1|cetux,-0.857980,-1.083032,G1,0,7.564151,5.307360


In [27]:
df_clean['uid']=df_clean['sample']+df_clean['trattamento']+'|'+df_clean['cell_id']+'|'+df_clean['trattamento']

In [28]:
df_clean.head()

,TNMD,C1orf112,CFTR,CYP26B1,ALS2,ARHGAP33,PDK4,PRSS22,MCUB,REXO5,...,H3C7,RDM1,H2AC4,C17orf78,H2BC10,H2AC17,sample,cell_id,trattamento,uid
0,1.086670,4.132281,5.390242,1.835423,3.574978,3.432078,4.246509,2.761224,4.947752,3.224731,...,1.549211,4.374855,0.850190,0.214806,0.850190,1.699403,CRC1502,AAACCCAAGGTGCGAT.1,cetux,CRC1502cetux|AAACCCAAGGTGCGAT.1|cetux
1,1.922616,2.807533,4.611481,3.621050,5.125645,2.210523,3.371584,4.417854,3.184945,2.450422,...,1.419355,0.504534,0.877611,1.173764,0.877611,0.188471,CRC1502,AAACCCACAACGGTAG.1,cetux,CRC1502cetux|AAACCCACAACGGTAG.1|cetux
2,4.584078,4.253414,5.064371,2.332159,3.823700,2.905356,1.860726,1.993791,3.976240,1.789288,...,0.610115,2.227945,0.767096,0.610115,1.037566,1.155892,CRC1502,AAACCCACAAGGTCAG.1,cetux,CRC1502cetux|AAACCCACAAGGTCAG.1|cetux
3,2.823299,5.147339,5.203333,1.738816,2.973977,3.968001,2.504677,2.095133,5.184909,3.717603,...,1.440553,4.913714,1.264577,0.208943,0.831232,0.952367,CRC1502,AAACCCACATTCAGGT.1,cetux,CRC1502cetux|AAACCCACATTCAGGT.1|cetux
4,3.118555,4.257699,5.322543,2.304837,3.589179,1.832576,2.789035,2.747305,3.243882,1.471709,...,0.670085,0.838527,0.989342,0.259490,1.250592,0.838527,CRC1502,AAACCCAGTATTGGCT.1,cetux,CRC1502cetux|AAACCCAGTATTGGCT.1|cetux


In [29]:
label=meta_data.loc[:,['uid','leiden_r0.3']]
label=label.rename(columns={'leiden_r0.3':'cluster'})

In [30]:
merged_df = df_clean.merge(label, on='uid', how='inner')


In [31]:
meta_cols = ['uid','sample','cell_id','trattamento','cluster']
feature_cols = [c for c in merged_df.columns if c not in meta_cols]  # geni/features numeriche

X = merged_df[feature_cols].copy()
y = merged_df['cluster'].copy()

# usa uid come index
X.index = merged_df['uid']
y.index = merged_df['uid']

# (OPZIONALE, ma spesso consigliato per scRNA-seq)
# split per campione per evitare
groups = merged_df.set_index('uid')['sample']  
use_group_split = False  # metti True se vuoi split per sample

if use_group_split:
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(X, y, groups=groups))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
else:
    # split classico ma STRATIFICATO (preserva proporzioni dei cluster)
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )


In [32]:
import pandas as pd

# oppure in forma tabellare percentuale
pd.concat([
    y_train.value_counts(normalize=True).rename('train'),
    y_test.value_counts(normalize=True).rename('test')
], axis=1).fillna(0).round(3)

train_uids = X_train.index
test_uids = X_test.index

sample_map = merged_df.set_index('uid')['sample']

# tabella riassuntiva
pd.concat([
    sample_map.loc[train_uids].value_counts(normalize=True).rename('train'),
    sample_map.loc[test_uids].value_counts(normalize=True).rename('test')
], axis=1).fillna(0).round(3)

train_meta = merged_df.set_index('uid').loc[train_uids, ['sample', 'cluster']]
test_meta = merged_df.set_index('uid').loc[test_uids, ['sample', 'cluster']]

print("\nTrain:")
display(pd.crosstab(train_meta['sample'], train_meta['cluster'], normalize='index').round(2))

print("\nTest:")
display(pd.crosstab(test_meta['sample'], test_meta['cluster'], normalize='index').round(2))



Train:


cluster,0,1,2,3,4,5,6
sample,,,,,,,
CRC1139,0.20,0.18,0.29,0.12,0.02,0.16,0.02
CRC1502,0.25,0.14,0.16,0.15,0.12,0.09,0.08
CRC1620,0.22,0.24,0.10,0.15,0.19,0.10,0.01



Test:


cluster,0,1,2,3,4,5,6
sample,,,,,,,
CRC1139,0.23,0.15,0.31,0.09,0.01,0.18,0.03
CRC1502,0.25,0.12,0.17,0.17,0.11,0.09,0.08
CRC1620,0.21,0.27,0.08,0.14,0.20,0.10,0.01


In [33]:
#scalo i dati
scaler = MaxAbsScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:

rf = RandomForestClassifier(random_state=42, n_jobs=3)

rf_grid = {
    'n_estimators': [300, 600],
    'max_depth': [None, 20, 40],
    'min_samples_split': [2, 5],
    'max_features': ['sqrt', 'log2']
}

rf_cv = GridSearchCV(
    rf,
    rf_grid,
    cv=3,
    scoring='f1_weighted',
    n_jobs=3,
    verbose=2
)

# Addestramento (X_train e X_test già scalati)
rf_cv.fit(X_train_scaled, y_train)

print("RF best params:", rf_cv.best_params_)
print("RF CV accuracy:", rf_cv.best_score_)

y_pred_rf = rf_cv.predict(X_test_scaled)
print("\n--- Random Forest Test ---")
print(classification_report(y_test, y_pred_rf))



Fitting 3 folds for each of 24 candidates, totalling 72 fits


/usr/local/mamba/envs/bbknn_env/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/local/mamba/envs/bbknn_env/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  return _ForkingPickler.loads(res)
/usr/local/mamba/envs/bbknn_env/lib/python3.10/multiprocessing/queues.py:122: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this p

In [16]:
data_path='/mnt/cold2/snaketree/prj/PPH/local/share/data/saver_mat'
test_id=['CRC0322','CRC0542','CRC0327']
tratt_cercato=['cetux','CTX72h']
test=pd.DataFrame()
for file in os.listdir(data_path):
    sample_name=str.split(file,sep='_')[4]
    trattamento=str.split(file,sep='_')[5]
    if sample_name in test_id and trattamento in tratt_cercato:
        print(file)
        data=pd.read_csv(os.path.join(data_path,file),header=0,index_col=0)
        data=data.T
        data['sample']=sample_name
        data['cell_id']=data.index
        data['trattamento']=trattamento
        data.reset_index(drop=True,inplace=True)
        test=pd.concat([test,data])

filtered_annotated_saver_ribomito_CRC0322_cetux_1_log2_pc1_cpm.csv
filtered_annotated_saver_ribomito_CRC0327_cetux_2_log2_pc1_cpm.csv
filtered_annotated_saver_ribomito_CRC0542_CTX72h_1_log2_pc1_cpm.csv


In [17]:
from func_NT_cetux import *
dt=test
df_clean_dt, dup = strip_prefix_from_genes(dt, meta_cols=("cell_id","sample","trattamento"), sep=":", on_duplicate="first")